# ME344 Option 1 — CPU, GH200, and TPU v5e Profiling Analysis

**Team:** ways-58  
**Author:** Stephanie Wang  
**Workload:** JAX/XLA multilayer perceptron for telecom churn classification

This notebook is the auditable analysis companion to the capstone. It reads the three selected 200-epoch benchmark JSON files committed under `results/`; it does not fabricate, smooth, or rerun measurements. Run the notebook from the repository root.

## 1. Experimental controls

The comparison holds the dataset and training configuration constant: 5,000 rows, 16 features, seed 344, stratified 70/15/15 train/validation/test split, batch size 256, float32, 200 epochs, and 2,600 measured training steps. Missing feature values are median-filled; standardization uses training-split mean and standard deviation. The DNN is `16 → 64 ReLU → 32 ReLU → 1 logit`, trained with binary cross-entropy and a JIT-compiled gradient-descent step.

The selected artifacts are the final portable CPU run, the immutable public-GHCR GH200 run, and the final eight-device TPU run with automated duty-cycle and HBM sampling. Verification runs and failed telemetry checks are excluded from the comparison.


In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent

RUN_FILES = {
    'Assigned CPU node': ROOT / 'results/cpu-ways-58-20260813-070716.json',
    'NVIDIA GH200': ROOT / 'results/churn-gpu-ways-58-20260813-074833.json',
    'TPU v5e 2x4': ROOT / 'results/churn-tpu-ways-58-telemetry-final2-20260813-211541.json',
}
runs = {name: json.loads(path.read_text()) for name, path in RUN_FILES.items()}
print('Loaded:', ', '.join(runs))


## 2. Comparability and provenance checks

The assertions below fail if a selected artifact is missing, malformed, or uses a different dataset size, feature count, epoch count, batch size, or step count.

In [ ]:
CONTROL_FIELDS = ('dataset_rows', 'features', 'epochs', 'batch_size', 'steps')
expected = {field: runs['Assigned CPU node'][field] for field in CONTROL_FIELDS}
assert expected == {
    'dataset_rows': 5000, 'features': 16, 'epochs': 200,
    'batch_size': 256, 'steps': 2600,
}
for platform, record in runs.items():
    observed = {field: record[field] for field in CONTROL_FIELDS}
    assert observed == expected, (platform, observed)
    assert record['telemetry']['device_count'] >= 1
print('PASS — all three selected runs use identical workload controls.')

## 3. Performance comparison

Training time isolates the 2,600 steady-state steps after the separately timed first JIT call. End-to-end time includes data loading, transfer, compilation, training, evaluation, and result preparation. Lower latency is better; higher steps/second is better.

In [ ]:
PERF_FIELDS = [
    ('compile_seconds', 'Compile s'),
    ('training_seconds', 'Training s'),
    ('measured_end_to_end_seconds', 'End-to-end s'),
    ('mean_step_ms', 'Mean step ms'),
    ('p95_step_ms', 'P95 step ms'),
    ('steps_per_second', 'Steps/s'),
]
header = ['Platform'] + [label for _, label in PERF_FIELDS]
rows = []
for platform, record in runs.items():
    rows.append([platform] + [record[key] for key, _ in PERF_FIELDS])
widths = [max(len(header[i]), *(len(f'{row[i]:.4f}') if isinstance(row[i], float) else len(str(row[i])) for row in rows)) for i in range(len(header))]
print(' | '.join(str(value).ljust(widths[i]) for i, value in enumerate(header)))
print('-+-'.join('-' * width for width in widths))
for row in rows:
    values = [row[0]] + [f'{value:.4f}' for value in row[1:]]
    print(' | '.join(value.ljust(widths[i]) for i, value in enumerate(values)))

In [ ]:
cpu = runs['Assigned CPU node']
gpu = runs['NVIDIA GH200']
tpu = runs['TPU v5e 2x4']
cpu_vs_gpu = gpu['training_seconds'] / cpu['training_seconds']
cpu_vs_tpu = tpu['training_seconds'] / cpu['training_seconds']
print(f'CPU training advantage vs GH200: {cpu_vs_gpu:.2f}x')
print(f'CPU training advantage vs TPU:   {cpu_vs_tpu:.2f}x')
assert round(cpu_vs_gpu, 2) == 3.12
assert round(cpu_vs_tpu, 2) == 1.42


### Presentation-ready performance charts

![Training time: CPU 4.97 s, GH200 15.49 s, TPU 7.04 s](../results/charts/training_time.svg)

![Throughput: CPU 523.23, GH200 167.84, TPU 369.12 steps per second](../results/charts/throughput.svg)


## 4. Predictive-quality comparison

Accuracy measures thresholded correctness at 0.5; ROC-AUC measures ranking quality across thresholds. Because the split and seed are fixed, cross-platform disagreement is retained and reported rather than normalized away.

In [ ]:
print(f"{'Platform':<20} {'Validation accuracy':>20} {'Test accuracy':>16} {'Test ROC-AUC':>14}")
print('-' * 74)
for platform, record in runs.items():
    print(f"{platform:<20} {record['validation_accuracy']:>20.4f} {record['test_accuracy']:>16.4f} {record['test_roc_auc']:>14.4f}")

![Accuracy and ROC-AUC comparison](../results/charts/predictive_quality.svg)

CPU and TPU ROC-AUC are nearly identical at approximately 0.693. GH200 accuracy is similar, but its ROC-AUC is lower at 0.623. Different JAX runtimes were unavoidable on the course platforms—CPU/TPU used JAX 0.6.2 while the NVIDIA course runtime supplied a 0.4.38 development build—so the mismatch is documented as a limitation and should be investigated before production use.

## 5. Device and telemetry evidence

The GPU was sampled with `nvidia-smi` every 0.5 seconds. CPU utilization is process CPU time divided by measured wall time and visible logical CPUs. The TPU container polled `tpu-info` every second for per-core duty cycle and per-device HBM. TPU mean duty cycle was 2.50% and peak duty cycle was 30.05%; only core 0 showed nonzero work in the final sample, demonstrating that the current implementation did not shard this small model across all eight devices. `tpu-info` reported 0.00 GiB used of 15.75 GiB per device; this observed value is retained but treated as a sampling/runtime limitation, not evidence of zero allocation.


In [ ]:
for platform, record in runs.items():
    telemetry = record['telemetry']
    print(f'\n{platform}')
    print('  devices:', telemetry['device_count'])
    print('  visible:', '; '.join(telemetry['visible_devices']))
    print('  JAX:', telemetry['jax_version'])
    print(f"  peak host RSS: {telemetry['peak_host_rss_mib']:.2f} MiB")
    if 'gpu_mean_utilization_percent' in telemetry:
        print(f"  GPU utilization mean/peak: {telemetry['gpu_mean_utilization_percent']:.1f}% / {telemetry['gpu_peak_utilization_percent']:.1f}%")
        print(f"  peak GPU memory: {telemetry['gpu_peak_memory_used_mib']:.0f} MiB")
    elif platform.startswith('TPU'):
        print(f"  TPU duty cycle mean/peak: {telemetry['tpu_mean_duty_cycle_percent']:.2f}% / {telemetry['tpu_peak_duty_cycle_percent']:.2f}%")
        print(f"  tpu-info HBM peak/total per device: {telemetry['tpu_peak_hbm_used_gib']:.2f} / {telemetry['tpu_hbm_total_gib_per_chip']:.2f} GiB")


## 6. Bottleneck diagnosis

The dominant issue is workload granularity, not accelerator capacity. A 5,000-row, 16-feature MLP performs too little dense-matrix work per synchronized step to amortize accelerator dispatch and synchronization. Evidence supporting this diagnosis:

- CPU achieved the lowest training and end-to-end times and the highest throughput.
- GH200 averaged only 1% sampled utilization, peaked at 2%, and used about 650 MiB of GPU memory.
- TPU averaged 2.50% duty cycle, peaked at 30.05%, and trained 1.42× slower than CPU despite exposing eight devices; the final sample showed work only on core 0.
- Data-loading and transfer times are only milliseconds in the formal runs, so CSV input is not the measured steady-state bottleneck.

The CPU mean step time is slightly above its P95 because a small number of slow outliers pull up the arithmetic mean; this is mathematically possible and the raw timing summary is retained.


## 7. Engineering and business recommendation

Use the assigned CPU class for the current periodic-retraining workload. It is fastest for the measured configuration and avoids reserving scarce accelerator capacity. Re-evaluate GPU/TPU only after materially increasing dataset size, model width/depth, batch size, retraining frequency, or concurrent jobs. Separately labeled scaling experiments could test larger batches, repeated/larger data, wider networks, and TPU bfloat16 without contaminating this float32 baseline.

For telecom retention, accuracy alone is not sufficient: the churn class may be operationally more important than the majority class. A deployment decision should select a threshold using retention-offer cost versus missed-churn loss, then monitor churn recall, precision, probability calibration, and feature/label drift. ROC-AUC near 0.69 indicates moderate ranking ability, so the model should not be described as production-ready without business validation.

## 8. Reproducibility notes and limitations

- Formal CPU artifact: `cpu-ways-58-20260813-070716.json`.
- Formal GPU artifact: `churn-gpu-ways-58-20260813-074833.json`, using the public GHCR image pinned by SHA-256 digest.
- Formal TPU artifact: `churn-tpu-ways-58-telemetry-final2-20260813-211541.json`, using Artifact Registry image digest `d39d826…f751f6c`.
- Earlier one-epoch checks, the superseded sklearn CPU run, GPU preallocation test, generic-NVIDIA-image runs, and failed TPU telemetry checks are excluded from the final matrix.
- GPU and CPU/TPU JAX versions differ, which limits strict numerical equivalence.
- GPU sampling is coarse relative to short kernels. TPU HBM was reported as 0.00 GiB by `tpu-info` and is documented as a telemetry limitation rather than interpreted literally.
- This is one run per selected platform, not a repeated-run confidence interval.

All conclusions above are scoped to the measured workload and should not be generalized to larger neural networks.
